In [53]:
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.optimizers import Adam
import pandas as pd
import tensorflow as tf
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import SGD, Adam , Nadam

import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,accuracy_score
from keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.utils import to_categorical

In [54]:
file_path = '/content/drive/MyDrive/Colab Notebooks/AKH_WQI.csv'
df = pd.read_csv(file_path)
df.head()

,PH,Temp,Turbidity,TSS,BOD5,COD,DO,Amoni,Phosphat,Coliforms,WQI
0,7.2,27.8,60.0,210.0,3.20,6.6,7.2,0.35,0.84,2600,49.68
1,7.1,27.6,65.0,180.0,3.15,6.8,7.7,0.88,0.95,4700,65.34
2,7.3,27.7,65.0,195.0,3.68,7.8,7.0,2.25,1.36,8500,51.53
3,7.1,27.5,70.0,180.0,3.50,6.2,6.7,2.33,1.41,7500,55.26
4,6.9,27.5,80.0,100.0,3.84,6.8,6.4,1.87,2.74,9500,49.51


In [55]:
scaler = StandardScaler()
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X = scaler.fit_transform(X)

In [56]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [57]:
model = Sequential()
model.add(Dense(units=128, activation="relu", input_shape=(X_train.shape[1],), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=64, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=32, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=1, activation="relu", kernel_regularizer=l2(0.01)))

model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error', metrics=['mae'])

model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_24 (Dense)            (None, 128)               1408      
                                                                 
 dropout_18 (Dropout)        (None, 128)               0         
                                                                 
 dense_25 (Dense)            (None, 64)                8256      
                                                                 
 dropout_19 (Dropout)        (None, 64)                0         
                                                                 
 dense_26 (Dense)            (None, 32)                2080      
                                                                 
 dropout_20 (Dropout)        (None, 32)                0         
                                                                 
 dense_27 (Dense)            (None, 1)                

In [58]:
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

model.fit(X_train, y_train, epochs=100, batch_size=20, validation_data=(X_val, y_val))

Epoch 1/100
23/23 [==============================] - 2s 24ms/step - loss: 1169.1536 - mae: 27.3851 - val_loss: 784.3421 - val_mae: 23.7979
Epoch 2/100
23/23 [==============================] - 0s 6ms/step - loss: 480.5780 - mae: 17.0060 - val_loss: 312.7021 - val_mae: 14.5380
Epoch 3/100
23/23 [==============================] - 0s 5ms/step - loss: 369.8627 - mae: 15.1512 - val_loss: 290.2536 - val_mae: 13.2786
Epoch 4/100
23/23 [==============================] - 0s 5ms/step - loss: 362.5857 - mae: 14.9584 - val_loss: 272.4971 - val_mae: 13.0209
Epoch 5/100
23/23 [==============================] - 0s 6ms/step - loss: 321.0463 - mae: 14.0822 - val_loss: 307.8026 - val_mae: 14.5702
Epoch 6/100
23/23 [==============================] - 0s 6ms/step - loss: 355.8994 - mae: 14.8216 - val_loss: 300.8928 - val_mae: 14.2317
Epoch 7/100
23/23 [==============================] - 0s 5ms/step - loss: 306.5150 - mae: 13.3625 - val_loss: 282.0363 - val_mae: 12.8957
Epoch 8/100
23/23 [====================

In [59]:
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {train_mse}")
print(f"Validation MSE: {val_mse}")
print(f"testing MSE: {test_mse}")

4/4 [==============================] - 0s 5ms/step
Train MSE: 97.01604302206314
Validation MSE: 261.28914526870943
testing MSE: 271.8676545938636


Với câu trúc này MSE(trainig) < MSE (testing) ==> Hight Variance(overfitting)

Thay đổi cấu trúc của mạng nơ-ron nhân tạo (ANN) bằng cách giảm số lượng nơ-ron trong một lớp thông qua Dropout và sử dụng L2 regularization (Ridge). Đồng thời, sử dụng phương pháp Early Stopping để ngừng huấn luyện sớm khi mô hình không còn cải thiện. Kết quả cho thấy MSE trên cả 3 tập có sự thay đổi tốt hơn, nhưng Train MSE (97.01) vẫn nhỏ hơn nhiều so với Testing MSE(271.86). Mặc dù kết quả đã được cải thiện đáng kể bằng cách sử dụng các phương pháp nhằm giảm variance, mô hình vẫn bị overfitting.


Thử thay đổi opitmizer SGD

In [60]:
model = Sequential()
model.add(Dense(units=128, activation="relu", input_shape=(X_train.shape[1],), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=64, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=32, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=1, activation="relu", kernel_regularizer=l2(0.01)))

model.compile(optimizer=SGD(learning_rate=0.01), loss='mean_squared_error', metrics=['mae'])

model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_28 (Dense)            (None, 128)               1408      
                                                                 
 dropout_21 (Dropout)        (None, 128)               0         
                                                                 
 dense_29 (Dense)            (None, 64)                8256      
                                                                 
 dropout_22 (Dropout)        (None, 64)                0         
                                                                 
 dense_30 (Dense)            (None, 32)                2080      
                                                                 
 dropout_23 (Dropout)        (None, 32)                0         
                                                                 
 dense_31 (Dense)            (None, 1)                

In [61]:
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

model.fit(X_train, y_train, epochs=100, batch_size=20, validation_data=(X_val, y_val))

Epoch 1/100
23/23 [==============================] - 1s 14ms/step - loss: 4694.0156 - mae: 46.3546 - val_loss: 4120.9326 - val_mae: 46.0540
Epoch 2/100
23/23 [==============================] - 0s 5ms/step - loss: 3654.4424 - mae: 41.2053 - val_loss: 4108.9321 - val_mae: 46.0540
Epoch 3/100
23/23 [==============================] - 0s 4ms/step - loss: 3642.4944 - mae: 41.2052 - val_loss: 4097.0415 - val_mae: 46.0540
Epoch 4/100
23/23 [==============================] - 0s 5ms/step - loss: 3630.6560 - mae: 41.2052 - val_loss: 4085.2600 - val_mae: 46.0540
Epoch 5/100
23/23 [==============================] - 0s 6ms/step - loss: 3618.9255 - mae: 41.2052 - val_loss: 4073.5864 - val_mae: 46.0540
Epoch 6/100
23/23 [==============================] - 0s 5ms/step - loss: 3607.3037 - mae: 41.2053 - val_loss: 4062.0198 - val_mae: 46.0540
Epoch 7/100
23/23 [==============================] - 0s 6ms/step - loss: 3595.7874 - mae: 41.2052 - val_loss: 4050.5591 - val_mae: 46.0540
Epoch 8/100
23/23 [=======

In [62]:
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {train_mse}")
print(f"Validation MSE: {val_mse}")
print(f"testing MSE: {test_mse}")

4/4 [==============================] - 0s 4ms/step
Train MSE: 2349.9139322440087
Validation MSE: 2810.66834040404
testing MSE: 2492.264584848485


khi thay đổi SGD opitmizer ta nhận thấy model không tốt với SGD, tiếp tục thay đổi thử với Nadam

In [63]:
model = Sequential()
model.add(Dense(units=128, activation="relu", input_shape=(X_train.shape[1],), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=64, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=32, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=1, activation="relu", kernel_regularizer=l2(0.01)))

model.compile(optimizer=Nadam(learning_rate=0.01), loss='mean_squared_error', metrics=['mae'])

model.summary()

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_32 (Dense)            (None, 128)               1408      
                                                                 
 dropout_24 (Dropout)        (None, 128)               0         
                                                                 
 dense_33 (Dense)            (None, 64)                8256      
                                                                 
 dropout_25 (Dropout)        (None, 64)                0         
                                                                 
 dense_34 (Dense)            (None, 32)                2080      
                                                                 
 dropout_26 (Dropout)        (None, 32)                0         
                                                                 
 dense_35 (Dense)            (None, 1)                

In [64]:
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

model.fit(X_train, y_train, epochs=100, batch_size=20, validation_data=(X_val, y_val))

Epoch 1/100
23/23 [==============================] - 2s 15ms/step - loss: 1209.0963 - mae: 26.9045 - val_loss: 328.3438 - val_mae: 14.6992
Epoch 2/100
23/23 [==============================] - 0s 5ms/step - loss: 408.4292 - mae: 16.0125 - val_loss: 291.1360 - val_mae: 13.7406
Epoch 3/100
23/23 [==============================] - 0s 5ms/step - loss: 375.2122 - mae: 15.1979 - val_loss: 291.9458 - val_mae: 14.1646
Epoch 4/100
23/23 [==============================] - 0s 5ms/step - loss: 365.5896 - mae: 15.0137 - val_loss: 272.7919 - val_mae: 12.8725
Epoch 5/100
23/23 [==============================] - 0s 6ms/step - loss: 342.9194 - mae: 14.6821 - val_loss: 313.0564 - val_mae: 14.3769
Epoch 6/100
23/23 [==============================] - 0s 7ms/step - loss: 347.9775 - mae: 14.4734 - val_loss: 306.2778 - val_mae: 14.2564
Epoch 7/100
23/23 [==============================] - 0s 6ms/step - loss: 311.5587 - mae: 13.6013 - val_loss: 294.9350 - val_mae: 13.9962
Epoch 8/100
23/23 [====================

In [65]:
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {train_mse}")
print(f"Validation MSE: {val_mse}")
print(f"testing MSE: {test_mse}")

4/4 [==============================] - 0s 4ms/step
Train MSE: 123.1034413327308
Validation MSE: 309.8252849014843
testing MSE: 260.4227678308558


Mặc dù đã áp dụng các kỹ thuật như Dropout và L2 regularization, mô hình vẫn có dấu hiệu overfitting. MSE trên tập huấn luyện nhỏ hơn nhiều so với trên tập validation và tập test giống như Adam. Nếu tiếp tục điều chỉnh các hyperparameters thì khả năng mô hình sẽ học tốt hơn

Chuyển qua bộ Thiên Ưu dataset

In [66]:
file_path = '/content/drive/MyDrive/Colab Notebooks/Thien_Uu.csv'
df = pd.read_csv(file_path)
df.head()

,GIST_0,GIST_1,GIST_2,GIST_3,GIST_4,GIST_5,GIST_6,GIST_7,GIST_8,GIST_9,...,GIST_119,GIST_120,GIST_121,GIST_122,GIST_123,GIST_124,GIST_125,GIST_126,GIST_127,class
0,0.001387,0.000820,0.001319,0.000786,0.001090,0.001229,0.001423,0.001787,0.001208,0.017221,...,0.001520,0.001119,0.001531,0.006161,0.002631,0.002506,0.033151,0.024942,0.002070,positive
1,0.014555,0.028314,0.026516,0.004162,0.001652,0.001647,0.004622,0.012038,0.002175,0.034284,...,0.006209,0.000769,0.001208,0.001088,0.000918,0.001917,0.011253,0.050512,0.004258,positive
2,0.002258,0.009734,0.001074,0.001196,0.001346,0.000902,0.001010,0.001091,0.003414,0.046807,...,0.001990,0.000785,0.003204,0.008465,0.007019,0.000827,0.013845,0.047319,0.001672,positive
3,0.000959,0.001216,0.002057,0.001346,0.001068,0.001529,0.001984,0.001013,0.001332,0.002051,...,0.002557,0.001676,0.000775,0.000469,0.000762,0.002117,0.028958,0.031339,0.001867,positive
4,0.001517,0.000951,0.000973,0.001367,0.001424,0.000976,0.001300,0.001189,0.001418,0.002512,...,0.003148,0.001907,0.003551,0.002413,0.003461,0.002297,0.022082,0.041486,0.001978,positive


In [67]:
X = df.drop(columns=['class'])
y = df['class']

le = LabelEncoder()
y = le.fit_transform(y)

In [68]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.fit_transform(X_test)

In [69]:
model = Sequential()
model.add(Dense(units=128, activation='relu', input_shape=(X_train.shape[1],),kernel_regularizer=l2(0.01)))
model.add(Dense(units=64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dense(units=32, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dense(units=1, activation='sigmoid', kernel_regularizer=l2(0.01)))

model.compile(optimizer = Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_36 (Dense)            (None, 128)               16512     
                                                                 
 dense_37 (Dense)            (None, 64)                8256      
                                                                 
 dense_38 (Dense)            (None, 32)                2080      
                                                                 
 dense_39 (Dense)            (None, 1)                 33        
                                                                 
Total params: 26881 (105.00 KB)
Trainable params: 26881 (105.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [70]:
call_back = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(X_train_scaled,
          y_train,
          epochs=200,
          batch_size=16,
          validation_data=(X_val_scaled, y_val),
          callbacks = [call_back],
          verbose = 1)

Epoch 1/200
88/88 [==============================] - 2s 10ms/step - loss: 1.0610 - accuracy: 0.8554 - val_loss: 0.5482 - val_accuracy: 0.8904
Epoch 2/200
88/88 [==============================] - 1s 6ms/step - loss: 0.5062 - accuracy: 0.8739 - val_loss: 0.4102 - val_accuracy: 0.9302
Epoch 3/200
88/88 [==============================] - 1s 16ms/step - loss: 0.4301 - accuracy: 0.8974 - val_loss: 0.4326 - val_accuracy: 0.8904
Epoch 4/200
88/88 [==============================] - 1s 12ms/step - loss: 0.3943 - accuracy: 0.9053 - val_loss: 0.3725 - val_accuracy: 0.9203
Epoch 5/200
88/88 [==============================] - 1s 11ms/step - loss: 0.3735 - accuracy: 0.9103 - val_loss: 0.3761 - val_accuracy: 0.9236
Epoch 6/200
88/88 [==============================] - 0s 6ms/step - loss: 0.3520 - accuracy: 0.9181 - val_loss: 0.3723 - val_accuracy: 0.9037
Epoch 7/200
88/88 [==============================] - 1s 6ms/step - loss: 0.3487 - accuracy: 0.9202 - val_loss: 0.3916 - val_accuracy: 0.9203
Epoch 8/2

In [71]:
train_loss, train_acc = model.evaluate(X_train_scaled, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val_scaled, y_val, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Acc on Training set: {train_acc * 100:2f}%")
print(f"Acc on validation set: {val_acc * 100:2f}%")
print(f"Acc on Test set: {test_acc * 100:2f}%")


Acc on Training set: 94.515669%
Acc on validation set: 93.687707%
Acc on Test set: 93.687707%


Dựa vào kết quả trên, ta thấy sự khác biệt giữa accuracy của tập Train và Test không quá lớn, khoảng 1-2%. Điều này cho thấy mô hình không có dấu hiệu overfitting rõ ràng. Việc thay đổi kiến trúc và sử dụng hợp lý các hyperparameter đã giúp đạt được độ chính xác của tập Train khá cao (94.51%) và độ chính xác của tập kiểm tra (93.68%). So sánh độ chính xác trên ba tập dữ liệu đều tương đối cao và không có sự chênh lệch lớn, do đó mô hình không có dấu hiệu của underfitting (high bias).

Với việc thay đổi kiến trúc của mạng ANN và việc sử dụng và thay đổi các giá trị của hyper parameter (learning rate, epochs và regularization,.....) thì mô hình sẽ dần đạt kết quả tốt nhất